# DPO Fine-tuning — India Steel Trade Intelligence Platform
**Model:** Qwen2.5-1.5B (4-bit quantised, fits on Colab T4 ~3GB VRAM)  
**Method:** Direct Preference Optimisation (DPO) via HuggingFace TRL  
**Dataset:** 70 preference pairs from `dpo/preference_pairs.json`  
**Goal:** Reduce over-hedging; increase citation specificity in Policy Analyst outputs  

Steps: install → load data → quantise model → LoRA → DPO train → eval → save checkpoint

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
!pip install -q trl transformers peft bitsandbytes accelerate datasets groq python-dotenv
!pip install -q sentence-transformers  # for faithfulness eval

In [ ]:
# ── 2. Upload preference pairs ─────────────────────────────────────────────────
# Either upload the file manually or paste the JSON directly
from google.colab import files
print('Upload dpo/preference_pairs.json from your local machine:')
uploaded = files.upload()  # select preference_pairs.json

In [ ]:
# ── 3. Load and format dataset ─────────────────────────────────────────────────
import json
from datasets import Dataset

with open('preference_pairs.json') as f:
    pairs = json.load(f)

print(f'Loaded {len(pairs)} preference pairs')

# DPOTrainer expects: prompt, chosen, rejected
# We format as chat messages using Qwen's chat template
SYSTEM_PROMPT = (
    'You are a steel trade policy analyst specialising in India. '
    'Answer questions about Indian steel trade policy using only the context provided. '
    'Always cite the specific regulation, investigation, or document that supports your answer. '
    'If you cannot find the answer in the provided context, say so explicitly.'
)

def format_pair(p):
    prompt = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': p['question']},
    ]
    chosen   = [{'role': 'assistant', 'content': p['chosen']}]
    rejected = [{'role': 'assistant', 'content': p['rejected']}]
    return {'prompt': prompt, 'chosen': chosen, 'rejected': rejected}

formatted = [format_pair(p) for p in pairs]

# 80/20 train/test split
split = int(len(formatted) * 0.8)
train_data = Dataset.from_list(formatted[:split])
test_data  = Dataset.from_list(formatted[split:])
print(f'Train: {len(train_data)}  |  Test: {len(test_data)}')

In [ ]:
# ── 4. Load model in 4-bit quantisation ───────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

print(f'Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ── 5. LoRA configuration ──────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 6. DPO Training ───────────────────────────────────────────────────────────
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    output_dir='./dpo_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    beta=0.1,                         # DPO temperature — controls deviation from base model
    max_length=1024,
    max_prompt_length=512,
    logging_steps=5,
    save_steps=20,
    evaluation_strategy='steps',
    eval_steps=20,
    bf16=True,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
)

print('Starting DPO training...')
trainer.train()
print('Training complete.')

In [ ]:
# ── 7. Save checkpoint ─────────────────────────────────────────────────────────
trainer.save_model('./dpo_checkpoint')
tokenizer.save_pretrained('./dpo_checkpoint')
print('Checkpoint saved to ./dpo_checkpoint')

# Zip and download
import subprocess
subprocess.run(['zip', '-r', 'dpo_checkpoint.zip', './dpo_checkpoint'])
files.download('dpo_checkpoint.zip')

In [ ]:
# ── 8. Faithfulness evaluation vs Groq baseline ───────────────────────────────
# Compare DPO model vs Groq llama-3.3-70b on the 10-question ground truth
# This is the gate test: if DPO faithfulness < Groq, keep Groq in production

EVAL_QUESTIONS = [
    'Which countries are subject to anti-dumping investigation on electrogalvanized steel imports into India?',
    'What products are covered under the anti-dumping investigation on seamless tubes from China?',
    'How many countries are named in the anti-dumping investigation on flat rolled products of stainless steel?',
    'What type of products are covered by the safeguard investigation on steel flat products?',
    'What does PCN stand for and why is it used in anti-dumping investigations?',
    'What is the difference between an anti-dumping measure and a safeguard measure?',
    'What is the IS 2062 standard and which steel products does it apply to?',
    'What triggers the initiation of a safeguard investigation in India?',
]

CONTEXT_PLACEHOLDER = (
    'Context: India uses DGTR for trade remedy investigations. '
    'Anti-dumping duties require material injury finding. '
    'Safeguard measures apply to all countries.'
)

print('Evaluating DPO model on eval questions...')
dpo_answers = []
for q in EVAL_QUESTIONS:
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Context:\n{CONTEXT_PLACEHOLDER}\n\nQuestion: {q}'},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    dpo_answers.append({'question': q, 'answer': answer})
    print(f'Q: {q[:60]}...')
    print(f'A: {answer[:150]}...')
    print()

# Save results
with open('dpo_eval_results.json', 'w') as f:
    json.dump(dpo_answers, f, indent=2)
print('Results saved to dpo_eval_results.json')

In [ ]:
# ── 9. NLI faithfulness comparison ───────────────────────────────────────────
# Compare DPO model faithfulness vs recorded Groq baseline (v3: faithfulness=1.00)
from sentence_transformers import CrossEncoder
import re

nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-small')

def nli_faith(context, answer):
    sentences = re.split(r'(?<=[.!?])\s+', answer.strip())
    sentences = [s for s in sentences if len(s) > 10]
    if not sentences:
        return 1.0
    pairs = [(context, s) for s in sentences]
    scores = nli_model.predict(pairs, apply_softmax=True)
    contradicted = sum(1 for s in scores if s.argmax() == 0)
    return round((len(sentences) - contradicted) / len(sentences), 3)

dpo_faithfulness = []
for item in dpo_answers:
    score = nli_faith(CONTEXT_PLACEHOLDER, item['answer'])
    dpo_faithfulness.append(score)
    print(f"NLI faith: {score:.3f}  |  {item['question'][:55]}")

avg_dpo = sum(dpo_faithfulness) / len(dpo_faithfulness)
groq_baseline = 1.00  # from eval/baseline_v3.csv avg faithfulness

print(f'\n=== GATE TEST ===')
print(f'DPO avg NLI faithfulness: {avg_dpo:.3f}')
print(f'Groq baseline (v3):       {groq_baseline:.3f}')
print(f'Delta: {avg_dpo - groq_baseline:+.3f}')
if avg_dpo >= groq_baseline - 0.05:
    print('DECISION: DPO model acceptable — use in production alongside Groq')
else:
    print('DECISION: DPO model below Groq threshold — keep Groq in production, document DPO result')

## Notes for Benchmark Report

Record the following in `eval/benchmark_v1_vs_v2_vs_v3.md`:
- DPO training loss (final epoch)
- DPO avg NLI faithfulness vs Groq baseline (1.00)
- Decision: keep Groq or deploy DPO
- Honest expectation: 70 pairs is a small dataset. Modest improvement (0.05–0.15) is the realistic range. A null result (keep Groq) is a valid documented finding per the plan.

As the plan states: *'The primary value is demonstrating DPO understanding, not achieving state-of-the-art calibration.'*